In [ ]:
import os
os.environ.setdefault("NUMBA_CACHE_DIR", "/tmp/numba_cache")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")

import warnings
warnings.filterwarnings("ignore")
import logging
# Suppress INFO logs, show only WARNING and above
logging.getLogger().setLevel(logging.WARNING)

from pathlib import Path

from repspat import SampleData, spatial_silhouette_analysis, spatial_constrained_hac, plot_spatial_clusters, pairwise_results_to_matrix, multiple_comparison, create_blocks

In [ ]:
import anndata as ad
import numpy as np

# Uses the AnnData produced by anndata_notebooks/visium_hd_binned_data.ipynb
candidate_paths = [
    Path("../data/visiumhd_human_colon_16um.h5ad"),
    Path("data/visiumhd_human_colon_16um.h5ad"),
]
adata_path = next((path for path in candidate_paths if path.exists()), None)
if adata_path is None:
    raise FileNotFoundError("Could not find visiumhd_human_colon_16um.h5ad in ../data or data.")

adata_full = ad.read_h5ad(adata_path, backed="r")

# Visium HD at 16um has many bins, so keep the case study notebook interactive.
rng = np.random.default_rng(0)
n_bins = min(2500, adata_full.n_obs)
obs = adata_full.obs.copy()

if "precomputed_cluster" in obs.columns:
    cluster_groups = obs.groupby("precomputed_cluster", observed=True).groups
    per_cluster = max(1, n_bins // len(cluster_groups))
    sampled = []
    for names in cluster_groups.values():
        names = np.array(list(names))
        take = min(per_cluster, len(names))
        sampled.extend(rng.choice(names, size=take, replace=False))

    if len(sampled) < n_bins:
        remaining = obs.index.difference(sampled).to_numpy()
        sampled.extend(rng.choice(remaining, size=n_bins - len(sampled), replace=False))

    sampled_obs = np.array(sampled[:n_bins])
else:
    sampled_obs = rng.choice(obs.index.to_numpy(), size=n_bins, replace=False)

adata = adata_full[sampled_obs, :].to_memory()
adata_full.file.close()

# Keep genes with broad signal across sampled bins.
n_genes = min(500, adata.n_vars)
gene_detection = np.asarray((adata.X > 0).sum(axis=0)).ravel()
top_genes = np.argsort(gene_detection)[-n_genes:]
adata = adata[:, top_genes].copy()

# Counts-per-10k normalization followed by log1p, preserving sparse storage until final densification.
X = adata.X.tocsr() if hasattr(adata.X, "tocsr") else adata.X
counts = np.asarray(X.sum(axis=1)).ravel()
scale = np.divide(1e4, counts, out=np.zeros_like(counts, dtype=float), where=counts > 0)
X = X.multiply(scale[:, None]) if hasattr(X, "multiply") else X * scale[:, None]
if hasattr(X, "data"):
    X.data = np.log1p(X.data)
    adata.X = X.toarray()
else:
    adata.X = np.log1p(X)

adata

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import pdist, squareform

class SpatialDataContainer:
    def __init__(self, adata, to_dense=True):
        # --- store raw AnnData ---
        self.sample_adata = adata

        # --- Feature matrix ---
        X = adata.X.toarray() if to_dense and hasattr(adata.X, "toarray") else adata.X

        self.feature_mat = pd.DataFrame(
            X,
            columns=adata.var_names,
            index=adata.obs_names
        )

        # --- Coordinates matrix ---
        self.coords_mat = pd.DataFrame(
            adata.obsm["spatial"],
            columns=["centroidX", "centroidY"],
            index=adata.obs_names
        )

        # --- Distance matrix (WARNING: O(n²) memory heavy) ---
        dist = squareform(pdist(self.feature_mat, metric="euclidean"))

        self.dist_matrix = pd.DataFrame(
            dist,
            index=self.feature_mat.index,
            columns=self.feature_mat.index
        )

In [ ]:
data = SpatialDataContainer(adata)

data.feature_mat
data.coords_mat
data.dist_matrix
data.sample_adata

In [ ]:
spatial_silhouette_analysis(data, n_neighbors_list=[8], n_clusters_range=range(8, 19))

In [ ]:
labels, feature_df, model = spatial_constrained_hac(data.sample_adata, feature_df=data.feature_mat, n_clusters=16, n_neighs=8)

In [ ]:
x = data.coords_mat.centroidX
y = data.coords_mat.centroidY

In [ ]:
plot_spatial_clusters(x, y, labels=labels, point_size=3, alpha=0.9, figsize=(8, 10))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

clusters = np.unique(labels)

ncols = 4
nrows = int(np.ceil(len(clusters) / ncols))

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(3*ncols, 4*nrows)
)

axes = axes.flatten()

for ax, cluster in zip(axes, clusters):
    mask = labels == cluster

    # background
    ax.scatter(
        x[~mask],
        y[~mask],
        c="lightgray",
        s=1,
        alpha=0.3
    )

    # cluster of interest
    ax.scatter(
        x[mask],
        y[mask],
        s=3
    )

    ax.set_title(f"Cluster {cluster}")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.invert_yaxis()

for ax in axes[len(clusters):]:
    ax.axis("off")

plt.tight_layout()
plt.show()